In [46]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/csiro-biomass/sample_submission.csv
/kaggle/input/csiro-biomass/train.csv
/kaggle/input/csiro-biomass/test.csv
/kaggle/input/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/csiro-biomass/train/ID2099464826.jpg
/kaggle/input/csiro-biomass/train/ID2037861084.jpg
/kaggle/input/csiro-biomass/train/ID1211362607.jpg
/kaggle/input/csiro-biomass/train/ID1853508321.jpg
/kaggle/input/csiro-biomass/train/ID193102215.jpg
/kaggle/input/csiro-biomass/train/ID698608346.jpg
/kaggle/input/csiro-biomass/train/ID1859251563.jpg
/kaggle/input/csiro-biomass/train/ID1880764911.jpg
/kaggle/input/csiro-biomass/train/ID853954911.jpg
/kaggle/input/csiro-biomass/train/ID1403107574.jpg
/kaggle/input/csiro-biomass/train/ID1781353117.jpg
/kaggle/input/csiro-biomass/train/ID384648061.jpg
/kaggle/input/csiro-biomass/train/ID1563418511.jpg
/kaggle/input/csiro-biomass/train/ID2125100696.jpg
/kaggle/input/csiro-biomass/train/ID482555369.jpg
/kaggle/input/csiro-biomass/train/ID638711343.jpg
/kaggle/input/c

In [47]:
# imports
import torch
from torch import nn
from torch import Tensor
import pandas as pd
import numpy as np
from PIL import Image

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, TensorDataset, Subset

import pytorch_lightning as pl
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import EarlyStopping

from torchmetrics import MeanAbsoluteError, R2Score

In [48]:
# set torch precision
torch.set_float32_matmul_precision('high')
# clear gpu cache
torch.cuda.empty_cache()

In [49]:
class ImageBiomassDataset(Dataset):
    def __init__(self, dataframe, transform=None, is_test=False):
        self.transform = transform
        self.is_test = is_test
        self.target_cols = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']

        if self.is_test:
            # TEST MODE:
            self.df = dataframe.groupby('image_path').first().reset_index()
        else:
            # TRAIN MODE:
            self.df = dataframe.pivot_table(
                index=['image_path'],
                columns='target_name',
                values='target'
            ).reset_index()
            # fill missing values, if any
            self.df[self.target_cols] = self.df[self.target_cols].fillna(0.0)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB') # get image from path
        if self.transform:
            image = self.transform(image)

        if self.is_test:
            # TEST MODE:
            image_id = row['image_path'].split('/')[-1].replace('.jpg', '')
            targets = torch.zeros(5) # dummy targets, not needed
        else:
            # TRAIN MODE:
            targets = torch.tensor([float(row[col]) for col in self.target_cols], dtype=torch.float32)
            image_id = "dummy_id" # dummy id, not needed
        return image, targets, image_id


In [50]:
class DeepConvNetImageOnly(LightningModule):
    def __init__(self, batch_size):
        super().__init__()
        self.batch_size = batch_size
        self.save_hyperparameters()
        self.model = self.build_model()
        self.register_buffer('target_weights', 
                             torch.tensor([0.1,0.1,0.1,0.5,0.2]))

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

    def forward(self, x):
        x = self.image_model(x)
        x = self.mlp(x)
        return x
    
    def loss(self, pred, target):
        residual_sos = self.target_weights * (target - pred)**2
        return residual_sos.mean()

    # training / validation loop
    def shared_step(self, mode:str, batch, batch_index):
        (image, targets, _) = batch
        x = image
        pred = self.forward(x)
        loss = self.loss(pred, targets)
        # just measure loss. if loss == 0 then r2 == 1 (perfect)
        self.log(f"{mode}_step_loss", loss, prog_bar=True, batch_size=self.batch_size)
        return loss

    def training_step(self, batch, batch_index):
        return self.shared_step('train', batch, batch_index)

    def validation_step(self, batch, batch_index):
        return self.shared_step('val', batch, batch_index)

    def test_step(self, batch, batch_index):
        return self.shared_step('test', batch, batch_index)

    def build_model(self):
        # images
        weights = torchvision.models.ResNet18_Weights.DEFAULT
        resnet = torchvision.models.resnet18(weights=None)
        weight_path = '/kaggle/input/resnet18/best_resnet18_model.pt'
        checkpoint = torch.load(weight_path, map_location='cpu', weights_only=False)
        if isinstance(checkpoint, dict):
            resnet.load_state_dict(checkpoint, strict=False) # load weights
        else:
            resnet = checkpoint
            
        self.image_model = nn.Sequential(
            *list(resnet.children())[:-1] # output: 512
        )
        # don't learn until the last layer (small model)
        for param in self.image_model.parameters():
            param.requires_grad = False

        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 5) # 5 targets
        )

In [51]:
# get data from csvs
data_path = "/kaggle/input/csiro-biomass/"
df_train = pd.read_csv(data_path + "train.csv")
df_test = pd.read_csv(data_path + "test.csv")
# fix the image paths
df_train['image_path'] = data_path + df_train['image_path']
df_test['image_path'] = data_path + df_test['image_path']

In [52]:
# define augmentation for expanding training set
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5), # 50% chance to flip
    transforms.RandomVerticalFlip(p=0.5),   # 50% chance to flip upside down
    transforms.RandomRotation(degrees=45),  # rotate +/- 45 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # change lighting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# define standard transformations for validation/testing
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [53]:
# prepare to make the datasets
unique_image_paths = df_train['image_path'].unique()
dataset_size = len(unique_image_paths)
indices = list(range(dataset_size))
split = int(np.floor(0.7 * dataset_size))
np.random.shuffle(indices)
train_paths = unique_image_paths[indices[:split]]
val_paths = unique_image_paths[indices[split:]]
df_train_split = df_train[df_train['image_path'].isin(train_paths)]
df_val_split = df_train[df_train['image_path'].isin(val_paths)]

In [54]:
# 2 training datasets, one with augmentation
train_dataset = ImageBiomassDataset(df_train_split, transform=train_transform, is_test=False)
val_dataset = ImageBiomassDataset(df_val_split, transform=val_transform, is_test=False)
test_dataset = ImageBiomassDataset(df_test, transform=val_transform, is_test=True)

In [55]:
# load the dataloaders
batch_size = 32
train_dataloader = DataLoader(dataset=train_dataset, batch_size=batch_size, num_workers=4, shuffle=True, drop_last=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=batch_size, num_workers=4, shuffle=False, drop_last=True)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=batch_size, num_workers=4, shuffle=False, drop_last=False)

In [56]:
# train the training module
model = DeepConvNetImageOnly(batch_size=batch_size)
epochs = 500
logger = CSVLogger('lightning_logs', name='my_lt_module')
early_stop = EarlyStopping(
    monitor='val_step_loss',
    min_delta=0.00,
    patience=20,
    verbose=True,
    mode='min'
)
trainer = Trainer(
    logger=logger,
    max_epochs=epochs,
    log_every_n_steps=1,
    accelerator='auto',
    callbacks=[early_stop],
)
trainer.fit(model=model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name        | Type       | Params | Mode 
---------------------------------------------------
0 | image_model | Sequential | 11.2 M | train
1 | mlp         | Sequential | 33.2 K | train
---------------------------------------------------
33.2 K    Trainable params
11.2 M    Non-trainable params
11.2 M    Total params
44.839    Total estimated model

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_step_loss improved. New best score: 346.017


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_step_loss did not improve in the last 20 records. Best score: 90.107. Signaling Trainer to stop.


In [61]:
# load the best model from checkpoints for the submission
# backpeddles 'patience' number of epochs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepConvNetImageOnly.load_from_checkpoint(
    trainer.checkpoint_callback.best_model_path,
    batch_size=batch_size
)
model.to(device)
print()

In [58]:
# manual testing the model
results = []
target_order = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']
model.eval()
with torch.no_grad():
    for batch in test_dataloader:
        (image, _, image_id) = batch
        image = image.to(device)
        pred = model(image).cpu().numpy()
        for i, img_id in enumerate(image_id):
            for target_id, target_name in enumerate(target_order):
                # remake the id 
                # ex: "ID12023__Dry_Clover_g"
                # also get the prediction
                sample_id = f"{img_id}__{target_name}"
                pred_value = max(0.0, pred[i][target_id])
                results.append({'sample_id': sample_id, 'target': pred_value})

In [59]:
# save the results to submission.csv
submission = pd.DataFrame(results)
submission.to_csv("submission.csv", index=False)
print('submission.csv saved')

submission.csv saved


In [60]:
submission

,sample_id,target
0,ID1001187975__Dry_Clover_g,4.724565
1,ID1001187975__Dry_Dead_g,8.385776
2,ID1001187975__Dry_Green_g,19.569530
3,ID1001187975__Dry_Total_g,32.360279
4,ID1001187975__GDM_g,24.226679
